# Retos · Katas

Katas cortas de lógica: conversiones, geometría, pequeños juegos con reglas y algún clásico de dos punteros o *backtracking*. Cada una se resuelve con una función pequeña y se comprueba con `assert` contra casos conocidos, la biblioteca estándar o una solución por fuerza bruta. Ideas de [retosdeprogramacion.com](https://retosdeprogramacion.com/ejercicios).

## 1. Relación de aspecto

Escribir `relacion_aspecto(ancho, alto)`, que devuelva la relación de aspecto simplificada de una resolución en forma de texto (`1920 × 1080` es `16:9`), dividiendo ambos lados entre su máximo común divisor. Comprobarlo con `fractions.Fraction`.

In [1]:
from fractions import Fraction
from math import gcd


def relacion_aspecto(ancho, alto):
    if ancho <= 0 or alto <= 0:
        raise ValueError("las dimensiones deben ser positivas")
    d = gcd(ancho, alto)
    return f"{ancho // d}:{alto // d}"


for ancho, alto in [(1920, 1080), (1280, 720), (1080, 1920), (2560, 1080), (1024, 768), (3440, 1440), (800, 800)]:
    r = relacion_aspecto(ancho, alto)
    a, b = map(int, r.split(":"))
    print(f"{ancho:>5} × {alto:<5} → {r}")
    assert Fraction(a, b) == Fraction(ancho, alto) and gcd(a, b) == 1

 1920 × 1080  → 16:9
 1280 × 720   → 16:9
 1080 × 1920  → 9:16
 2560 × 1080  → 64:27
 1024 × 768   → 4:3
 3440 × 1440  → 43:18
  800 × 800   → 1:1


## 2. Ley de Ohm

Escribir `ley_ohm(v=None, r=None, i=None)`, que reciba exactamente dos de las tres magnitudes (tensión en voltios, resistencia en ohmios, intensidad en amperios) y devuelva un texto con la que falta, con dos decimales. Si no llegan exactamente dos datos o la resistencia no es positiva, devolver `"Datos inválidos"`.

In [2]:
import math


def ley_ohm(v=None, r=None, i=None):
    datos = {"V": v, "R": r, "I": i}
    falta = [k for k, x in datos.items() if x is None]
    if len(falta) != 1 or (r is not None and r <= 0):
        return "Datos inválidos"
    match falta[0]:
        case "V":
            return f"V = {r * i:.2f} V"
        case "R":
            return f"R = {v / i:.2f} Ω" if i else "Datos inválidos"
        case "I":
            return f"I = {v / r:.2f} A"


casos = [dict(r=220, i=0.05), dict(v=12, i=0.5), dict(v=230, r=46), dict(v=5), dict(v=1, r=2, i=3),
         dict(v=9, r=0), dict(v=9, i=0)]
for caso in casos:
    print(f"{str(caso):<30} {ley_ohm(**caso)}")

for v, r in [(12, 4), (230, 46), (3.3, 220), (5, 1000)]:
    i = float(ley_ohm(v=v, r=r).split()[2])
    assert math.isclose(i, round(v / r, 2))
assert ley_ohm(r=220, i=0.05) == "V = 11.00 V"
assert all(ley_ohm(**c) == "Datos inválidos" for c in casos[3:])

{'r': 220, 'i': 0.05}          V = 11.00 V
{'v': 12, 'i': 0.5}            R = 24.00 Ω
{'v': 230, 'r': 46}            I = 5.00 A
{'v': 5}                       Datos inválidos
{'v': 1, 'r': 2, 'i': 3}       Datos inválidos
{'v': 9, 'r': 0}               Datos inválidos
{'v': 9, 'i': 0}               Datos inválidos


## 3. Vectores ortogonales

Dos vectores son ortogonales si su producto escalar es cero. Escribir `son_ortogonales(u, v)` para vectores de cualquier dimensión (con `math.sumprod`), con una tolerancia para los decimales, y que lance `ValueError` si las dimensiones no coinciden. Comprobar los resultados con `numpy.dot` y con el ángulo que forman.

In [3]:
import numpy as np


def son_ortogonales(u, v, tol=1e-9):
    if len(u) != len(v):
        raise ValueError(f"dimensiones distintas: {len(u)} y {len(v)}")
    return abs(math.sumprod(u, v)) <= tol


pares = [((1, 2), (2, 1)), ((1, 2), (-2, 1)), ((3, -1, 2), (1, 1, -1)), ((0.5, 0.25), (-1, 2)), ((1, 0, 0, 1), (0, 1, 1, 0))]
for u, v in pares:
    angulo = math.degrees(math.acos(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))))
    print(f"{str(u):<16} · {str(v):<16} = {math.sumprod(u, v):>5} → {'ortogonales' if son_ortogonales(u, v) else 'no':<11} ({angulo:.1f}°)")
    assert son_ortogonales(u, v) == math.isclose(angulo, 90) == np.isclose(np.dot(u, v), 0)

try:
    son_ortogonales((1, 2), (1, 2, 3))
except ValueError as e:
    print("Error:", e)

(1, 2)           · (2, 1)           =     4 → no          (36.9°)
(1, 2)           · (-2, 1)          =     0 → ortogonales (90.0°)
(3, -1, 2)       · (1, 1, -1)       =     0 → ortogonales (90.0°)
(0.5, 0.25)      · (-1, 2)          =   0.0 → ortogonales (90.0°)
(1, 0, 0, 1)     · (0, 1, 1, 0)     =     0 → ortogonales (90.0°)
Error: dimensiones distintas: 2 y 3


## 4. A milisegundos

Escribir `a_milisegundos(dias=0, horas=0, minutos=0, segundos=0)`, que convierta una duración a milisegundos multiplicando por los factores adecuados. Comprobarla con `datetime.timedelta` en varias duraciones aleatorias.

In [4]:
import random
from datetime import timedelta

MS = {"dias": 24 * 60 * 60 * 1000, "horas": 60 * 60 * 1000, "minutos": 60 * 1000, "segundos": 1000}


def a_milisegundos(dias=0, horas=0, minutos=0, segundos=0):
    return dias * MS["dias"] + horas * MS["horas"] + minutos * MS["minutos"] + segundos * MS["segundos"]


print("Un día:", f"{a_milisegundos(dias=1):_}".replace("_", "."), "ms")
print("La travesía Los Cristianos–San Sebastián (50 min):", f"{a_milisegundos(minutos=50):_}".replace("_", "."), "ms")
print("2 d 3 h 4 min 5 s:", f"{a_milisegundos(2, 3, 4, 5):_}".replace("_", "."), "ms")

random.seed(4)
for _ in range(1000):
    d, h, m, s = (random.randint(0, 400) for _ in range(4))
    assert a_milisegundos(d, h, m, s) == timedelta(days=d, hours=h, minutes=m, seconds=s) // timedelta(milliseconds=1)

Un día: 86.400.000 ms
La travesía Los Cristianos–San Sebastián (50 min): 3.000.000 ms
2 d 3 h 4 min 5 s: 183.845.000 ms


## 5. Colores HEX ↔ RGB

Escribir `hex_a_rgb(texto)`, que acepte `#RRGGBB` y la forma corta `#RGB` (cada cifra se duplica: `#FA0` es `#FFAA00`) y devuelva una tupla `(r, g, b)`, y `rgb_a_hex(r, g, b)`, que valide que cada canal está entre 0 y 255. Comprobar la ida y vuelta con mil colores aleatorios.

In [5]:
import re


def hex_a_rgb(texto):
    if not re.fullmatch(r"#([0-9a-fA-F]{3}|[0-9a-fA-F]{6})", texto):
        raise ValueError(f"color HEX no válido: {texto!r}")
    cifras = texto[1:]
    if len(cifras) == 3:
        cifras = "".join(c * 2 for c in cifras)
    return tuple(int(cifras[i:i + 2], 16) for i in (0, 2, 4))


def rgb_a_hex(r, g, b):
    if not all(isinstance(c, int) and 0 <= c <= 255 for c in (r, g, b)):
        raise ValueError(f"canal fuera de rango: {(r, g, b)}")
    return f"#{r:02X}{g:02X}{b:02X}"


colores = {"azul océano": "#0077BE", "amarillo canario": "#FFEF00", "verde tabaiba": "#6B8E23", "naranja": "#FA0"}
for nombre, hexa in colores.items():
    rgb = hex_a_rgb(hexa)
    print(f"{nombre:<17} {hexa:<8} → rgb{rgb} → {rgb_a_hex(*rgb)}")

for malo in ["0077BE", "#12345", "#GG0000"]:
    try:
        hex_a_rgb(malo)
    except ValueError as e:
        print("Error:", e)
try:
    rgb_a_hex(256, 0, 0)
except ValueError as e:
    print("Error:", e)

random.seed(5)
for _ in range(1000):
    rgb = tuple(random.randint(0, 255) for _ in range(3))
    assert hex_a_rgb(rgb_a_hex(*rgb)) == rgb
    assert hex_a_rgb(rgb_a_hex(*rgb).lower()) == rgb
assert hex_a_rgb("#FA0") == hex_a_rgb("#FFAA00") == (255, 170, 0)

azul océano       #0077BE  → rgb(0, 119, 190) → #0077BE
amarillo canario  #FFEF00  → rgb(255, 239, 0) → #FFEF00
verde tabaiba     #6B8E23  → rgb(107, 142, 35) → #6B8E23
naranja           #FA0     → rgb(255, 170, 0) → #FFAA00
Error: color HEX no válido: '0077BE'
Error: color HEX no válido: '#12345'
Error: color HEX no válido: '#GG0000'
Error: canal fuera de rango: (256, 0, 0)


## 6. El robot que gira

Un robot empieza en `(0, 0)` mirando al norte (eje $y$ positivo). Recibe una lista de pasos: en cada uno avanza esa cantidad (hacia atrás si es negativa) y después gira 90° a la izquierda. Escribir `posicion_final(pasos)` con una tabla de direcciones y comprobarla con números complejos, donde girar a la izquierda es multiplicar por $i$.

In [6]:
DIRECCIONES = [(0, 1), (-1, 0), (0, -1), (1, 0)]  # norte, oeste, sur, este: giro a la izquierda


def posicion_final(pasos):
    x = y = 0
    d = 0
    for n in pasos:
        dx, dy = DIRECCIONES[d]
        x, y = x + n * dx, y + n * dy
        d = (d + 1) % 4
    return x, y


def posicion_compleja(pasos):
    posicion, direccion = 0, 1j
    for n in pasos:
        posicion += n * direccion
        direccion *= 1j
    return int(posicion.real), int(posicion.imag)


for pasos in [[10, 5, -2], [3, 3, 3, 3], [0, 0, 0, 1], [-5, 7, 2, 8, -1, 4]]:
    print(f"{str(pasos):<22} → {posicion_final(pasos)}")

random.seed(6)
for _ in range(500):
    pasos = [random.randint(-20, 20) for _ in range(random.randint(0, 15))]
    assert posicion_final(pasos) == posicion_compleja(pasos)
assert posicion_final([7] * 4) == (0, 0)

[10, 5, -2]            → (-5, 12)
[3, 3, 3, 3]           → (0, 0)
[0, 0, 0, 1]           → (1, 0)
[-5, 7, 2, 8, -1, 4]   → (-3, -8)


## 7. Pista de obstáculos

Una pista es un texto con suelo (`_`) y vallas (`|`), y un atleta recibe una lista de acciones `"correr"` o `"saltar"`, una por tramo. Escribir `recorrer(acciones, pista)`, que devuelva la pista recorrida y si la ha superado: correr en suelo deja `_`, saltar una valla deja `|`, saltar en el suelo se marca con `x` y correr contra una valla con `/`. Si hay más o menos acciones que tramos, los tramos sobrantes se marcan con `?` y la pista no se supera.

In [7]:
from itertools import zip_longest

MARCAS = {("correr", "_"): "_", ("saltar", "|"): "|", ("saltar", "_"): "x", ("correr", "|"): "/"}


def recorrer(acciones, pista):
    resultado = "".join(MARCAS.get((a, t), "?") for a, t in zip_longest(acciones, pista))
    return resultado, resultado == pista


pista = "__|_|__|_"
pruebas = {
    "perfecta": ["correr", "correr", "saltar", "correr", "saltar", "correr", "correr", "saltar", "correr"],
    "dos fallos": ["correr", "saltar", "saltar", "correr", "correr", "correr", "correr", "saltar", "correr"],
    "se queda corta": ["correr", "correr", "saltar", "correr"],
    "le sobran": ["correr", "correr", "saltar", "correr", "saltar", "correr", "correr", "saltar", "correr", "correr"],
}
print(f"{'pista':<15} {pista}")
for nombre, acciones in pruebas.items():
    recorrida, superada = recorrer(acciones, pista)
    print(f"{nombre:<15} {recorrida:<11} {'superada' if superada else 'no superada'}")

assert recorrer(pruebas["perfecta"], pista) == (pista, True)
assert recorrer(pruebas["dos fallos"], pista) == ("_x|_/__|_", False)
assert recorrer(pruebas["se queda corta"], pista)[0] == "__|_?????"
assert recorrer(pruebas["le sobran"], pista)[0] == pista + "?"

pista           __|_|__|_
perfecta        __|_|__|_   superada
dos fallos      _x|_/__|_   no superada
se queda corta  __|_?????   no superada
le sobran       __|_|__|_?  no superada


## 8. Marcador de tenis

Escribir `marcador_tenis(puntos, j1, j2)`, que reciba la secuencia de quién gana cada punto de un juego y devuelva el marcador tras cada punto: `0`, `15`, `30`, `40`; `Iguales` si ambos llegan a 40 con los mismos puntos; `Ventaja para …` si uno va un punto por delante a partir de ahí, y `Juego para …` cuando uno tiene al menos 4 puntos y 2 de ventaja. Lanzar `ValueError` si llegan puntos con el juego terminado o de un jugador desconocido.

In [8]:
NOMBRES_PUNTOS = ["0", "15", "30", "40"]


def marcador_tenis(puntos, j1="Nira", j2="Tomás"):
    a = b = 0
    marcadores = []
    for n, p in enumerate(puntos, 1):
        if p not in (j1, j2):
            raise ValueError(f"punto {n}: jugador desconocido {p!r}")
        if marcadores and marcadores[-1].startswith("Juego"):
            raise ValueError(f"punto {n}: el juego ya ha terminado")
        a, b = (a + 1, b) if p == j1 else (a, b + 1)
        lider = j1 if a > b else j2
        if max(a, b) >= 4 and abs(a - b) >= 2:
            marcadores.append(f"Juego para {lider}")
        elif a >= 3 and b >= 3:
            marcadores.append("Iguales" if a == b else f"Ventaja para {lider}")
        else:
            marcadores.append(f"{NOMBRES_PUNTOS[a]} - {NOMBRES_PUNTOS[b]}")
    return marcadores


puntos = ["Nira", "Nira", "Tomás", "Tomás", "Nira", "Tomás", "Tomás", "Nira", "Nira", "Nira"]
for p, m in zip(puntos, marcador_tenis(puntos)):
    print(f"punto de {p:<6} → {m}")

for malos in [puntos + ["Tomás"], ["Nira", "Ana"]]:
    try:
        marcador_tenis(malos)
    except ValueError as e:
        print("Error:", e)

random.seed(8)
for _ in range(2000):
    secuencia, a, b = [], 0, 0
    while not (max(a, b) >= 4 and abs(a - b) >= 2):
        p = random.choice(["Nira", "Tomás"])
        secuencia.append(p)
        a, b = secuencia.count("Nira"), secuencia.count("Tomás")
    marcadores = marcador_tenis(secuencia)
    assert marcadores[-1] == f"Juego para {'Nira' if a > b else 'Tomás'}"
    assert not any(m.startswith("Juego") for m in marcadores[:-1])
    for k, m in enumerate(marcadores[:-1], 1):
        ak, bk = secuencia[:k].count("Nira"), secuencia[:k].count("Tomás")
        assert (m == "Iguales") == (ak == bk >= 3)

punto de Nira   → 15 - 0
punto de Nira   → 30 - 0
punto de Tomás  → 30 - 15
punto de Tomás  → 30 - 30
punto de Nira   → 40 - 30
punto de Tomás  → Iguales
punto de Tomás  → Ventaja para Tomás
punto de Nira   → Iguales
punto de Nira   → Ventaja para Nira
punto de Nira   → Juego para Nira
Error: punto 11: el juego ya ha terminado
Error: punto 2: jugador desconocido 'Ana'


## 9. Piedra, papel, tijera… lagarto, Spock

Guardar las reglas en un diccionario `GANA_A` que asocie a cada jugada el conjunto de jugadas a las que vence, para la versión de cinco jugadas (piedra, papel, tijera, lagarto y Spock), y obtener de él la versión clásica de tres. Escribir `ganador(rondas, reglas)`, que reciba una lista de rondas `(jugada1, jugada2)` y devuelva `"Jugador 1"`, `"Jugador 2"` o `"Empate"` según quién gane más rondas. Comprobar que las reglas son un torneo justo: en cada pareja de jugadas distintas gana exactamente una y todas vencen al mismo número de jugadas.

In [9]:
GANA_A = {
    "piedra": {"tijera", "lagarto"},
    "papel": {"piedra", "spock"},
    "tijera": {"papel", "lagarto"},
    "lagarto": {"papel", "spock"},
    "spock": {"piedra", "tijera"},
}
CLASICAS = {"piedra", "papel", "tijera"}
GANA_A_CLASICO = {j: GANA_A[j] & CLASICAS for j in CLASICAS}


def ganador(rondas, reglas=GANA_A):
    victorias = [0, 0]
    for j1, j2 in rondas:
        if j1 not in reglas or j2 not in reglas:
            raise ValueError(f"jugada no válida en {(j1, j2)}")
        if j2 in reglas[j1]:
            victorias[0] += 1
        elif j1 in reglas[j2]:
            victorias[1] += 1
    if victorias[0] == victorias[1]:
        return "Empate"
    return "Jugador 1" if victorias[0] > victorias[1] else "Jugador 2"


partidas = [
    ([("piedra", "tijera"), ("tijera", "piedra"), ("papel", "tijera")], GANA_A_CLASICO),
    ([("piedra", "piedra"), ("papel", "piedra")], GANA_A_CLASICO),
    ([("spock", "lagarto"), ("lagarto", "papel"), ("piedra", "spock"), ("tijera", "lagarto")], GANA_A),
    ([("spock", "spock"), ("lagarto", "piedra"), ("papel", "tijera")], GANA_A),
]
for rondas, reglas in partidas:
    print(f"{len(reglas)} jugadas, {len(rondas)} rondas → {ganador(rondas, reglas)}")
try:
    ganador([("lagarto", "piedra")], GANA_A_CLASICO)
except ValueError as e:
    print("Error:", e)

for reglas in (GANA_A, GANA_A_CLASICO):
    for a in reglas:
        for b in reglas:
            if a != b:
                assert (b in reglas[a]) != (a in reglas[b])
    assert {len(v) for v in reglas.values()} == {(len(reglas) - 1) // 2}
assert [ganador(r, g) for r, g in partidas] == ["Jugador 2", "Jugador 1", "Empate", "Jugador 2"]

3 jugadas, 3 rondas → Jugador 2
3 jugadas, 2 rondas → Jugador 1
5 jugadas, 4 rondas → Empate
5 jugadas, 3 rondas → Jugador 2
Error: jugada no válida en ('lagarto', 'piedra')


## 10. Leer un ábaco

Un ábaco de 7 varillas se describe con una lista de cadenas, de las unidades de millón a las unidades. Cada varilla tiene 9 cuentas `O` y un hueco `---`: las cuentas a la izquierda del hueco forman la cifra. Escribir `leer_abaco(varillas)`, que valide el formato y devuelva el número, y `dibujar_abaco(n)`, que haga lo contrario. Comprobar la ida y vuelta.

In [10]:
def leer_abaco(varillas):
    numero = 0
    for v in varillas:
        if v.count("O") != 9 or v.count("---") != 1 or set(v) - {"O", "-"} or len(v) != 12:
            raise ValueError(f"varilla mal formada: {v!r}")
        numero = numero * 10 + v.index("---")
    return numero


def dibujar_abaco(n, varillas=7):
    if not 0 <= n < 10**varillas:
        raise ValueError(f"{n} no cabe en {varillas} varillas")
    return ["O" * int(c) + "---" + "O" * (9 - int(c)) for c in f"{n:0{varillas}d}"]


abaco = [
    "O---OOOOOOOO",
    "OOO---OOOOOO",
    "---OOOOOOOOO",
    "OO---OOOOOOO",
    "OOOOOOO---OO",
    "OOOOOOOOO---",
    "OOOOO---OOOO",
]
print("\n".join(abaco))
print("Número:", f"{leer_abaco(abaco):_}".replace("_", "."))

try:
    leer_abaco(["OO--OOOOOOO"])
except ValueError as e:
    print("Error:", e)

assert leer_abaco(abaco) == 1_302_795 and dibujar_abaco(1_302_795) == abaco
random.seed(10)
for n in [0, 9_999_999] + random.sample(range(10**7), 1000):
    assert leer_abaco(dibujar_abaco(n)) == n

O---OOOOOOOO
OOO---OOOOOO
---OOOOOOOOO
OO---OOOOOOO
OOOOOOO---OO
OOOOOOOOO---
OOOOO---OOOO
Número: 1.302.795
Error: varilla mal formada: 'OO--OOOOOOO'


## 11. Espiral con caracteres de caja

Dibujar una espiral cuadrada de lado `n` que ocupe todas las casillas, empezando por la esquina superior izquierda hacia la derecha y girando siempre en sentido horario hacia dentro. Usar `═` y `║` para los tramos rectos y `╗ ╝ ╚ ╔` para los giros; el carácter de cada casilla depende de la dirección con la que se llega y con la que se sale. Comprobar que todas las casillas se usan y que hay $2n - 1$ esquinas.

In [11]:
DERECHA, ABAJO, IZQUIERDA, ARRIBA = (0, 1), (1, 0), (0, -1), (-1, 0)
HORARIO = [DERECHA, ABAJO, IZQUIERDA, ARRIBA]
GIROS = {(DERECHA, ABAJO): "╗", (ABAJO, IZQUIERDA): "╝", (IZQUIERDA, ARRIBA): "╚", (ARRIBA, DERECHA): "╔"}


def espiral(n):
    rejilla = [[" "] * n for _ in range(n)]
    recorrido, f, c, d = [], 0, 0, 0
    for _ in range(n * n):
        recorrido.append((f, c, HORARIO[d]))
        rejilla[f][c] = "·"
        df, dc = HORARIO[d]
        if not (0 <= f + df < n and 0 <= c + dc < n and rejilla[f + df][c + dc] == " "):
            d = (d + 1) % 4
            df, dc = HORARIO[d]
        f, c = f + df, c + dc
    # la última casilla dibuja el giro que haría a continuación
    for k, (f, c, entra) in enumerate(recorrido):
        sale = recorrido[k + 1][2] if k + 1 < len(recorrido) else HORARIO[(HORARIO.index(entra) + 1) % 4]
        rejilla[f][c] = GIROS.get((entra, sale)) or ("═" if entra in (DERECHA, IZQUIERDA) else "║")
    return ["".join(fila) for fila in rejilla]


for n in (5, 8):
    print("\n".join(espiral(n)), end="\n\n")

assert espiral(5) == ["════╗", "╔══╗║", "║╔╗║║", "║╚═╝║", "╚═══╝"]
for n in range(1, 30):
    dibujo = "".join(espiral(n))
    assert " " not in dibujo and len(dibujo) == n * n
    assert sum(dibujo.count(g) for g in "╗╝╚╔") == 2 * n - 1

════╗
╔══╗║
║╔╗║║
║╚═╝║
╚═══╝

═══════╗
╔═════╗║
║╔═══╗║║
║║╔═╗║║║
║║║╚╝║║║
║║╚══╝║║
║╚════╝║
╚══════╝



## 12. Agua de lluvia atrapada

Unos bloques de ancho 1 tienen las alturas de una lista. Tras la lluvia, sobre cada bloque se queda tanta agua como el mínimo entre el bloque más alto a su izquierda y el más alto a su derecha, menos su altura. Calcular el agua total con dos punteros que avanzan desde los extremos moviendo siempre el lado más bajo ($O(n)$ y memoria $O(1)$), comprobarlo con la fuerza bruta $O(n^2)$ y dibujar el resultado.

In [12]:
def agua_dos_punteros(alturas):
    i, j = 0, len(alturas) - 1
    max_izq = max_der = total = 0
    while i < j:
        if alturas[i] <= alturas[j]:
            max_izq = max(max_izq, alturas[i])
            total += max_izq - alturas[i]
            i += 1
        else:
            max_der = max(max_der, alturas[j])
            total += max_der - alturas[j]
            j -= 1
    return total


def agua_por_bloque(alturas):
    return [max(0, min(max(alturas[:k + 1]), max(alturas[k:])) - h) for k, h in enumerate(alturas)]


def dibujar(alturas):
    agua = agua_por_bloque(alturas)
    for nivel in range(max(alturas), 0, -1):
        print("".join("#" if h >= nivel else "~" if h + a >= nivel else " " for h, a in zip(alturas, agua)))


bloques = [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1, 0, 4, 0, 2]
dibujar(bloques)
print("Agua atrapada:", agua_dos_punteros(bloques))

assert agua_dos_punteros(bloques) == sum(agua_por_bloque(bloques)) == 16
assert agua_dos_punteros([]) == agua_dos_punteros([5]) == agua_dos_punteros([1, 2, 3]) == 0
random.seed(12)
for _ in range(2000):
    alturas = [random.randint(0, 8) for _ in range(random.randint(0, 20))]
    assert agua_dos_punteros(alturas) == sum(agua_por_bloque(alturas))

             #  
       #~~~~~#  
   #~~~##~#~~#~#
 #~##~######~#~#
Agua atrapada: 16


## 13. Pagar justo con las monedas de la hucha

Una hucha tiene monedas repetidas y se quiere pagar exactamente una cantidad usando cada moneda como mucho una vez. Listar todas las formas distintas de hacerlo (dos formas con los mismos valores son la misma aunque usen monedas físicas distintas). En `09-recursividad` se resolvió saltando repetidos en cada nivel; aquí el *backtracking* recorre un `Counter` de valores y decide cuántas monedas usar de cada uno, así que no puede generar duplicados. Comprobarlo con `itertools.combinations`.

In [13]:
from collections import Counter
from itertools import combinations


def formas_de_pagar(monedas, objetivo):
    valores = sorted(Counter(monedas).items())
    soluciones = []

    def explorar(k, resto, elegidas):
        if resto == 0:
            soluciones.append(tuple(elegidas))
            return
        if k == len(valores) or resto < valores[k][0]:
            return
        valor, cuantas = valores[k]
        for usadas in range(min(cuantas, resto // valor), -1, -1):
            explorar(k + 1, resto - usadas * valor, elegidas + [valor] * usadas)

    explorar(0, objetivo, [])
    return sorted(soluciones)


def fuerza_bruta(monedas, objetivo):
    return sorted({tuple(sorted(c)) for r in range(len(monedas) + 1) for c in combinations(monedas, r) if sum(c) == objetivo})


hucha = [50, 20, 20, 10, 10, 10, 5, 5, 2, 1, 1]  # céntimos
for forma in formas_de_pagar(hucha, 37):
    print(" + ".join(map(str, forma)), "= 37")

assert formas_de_pagar(hucha, 37) == fuerza_bruta(hucha, 37)
assert formas_de_pagar(hucha, 0) == [()] and formas_de_pagar(hucha, 1000) == []
random.seed(13)
for _ in range(300):
    monedas = [random.randint(1, 9) for _ in range(random.randint(0, 11))]
    objetivo = random.randint(0, 25)
    assert formas_de_pagar(monedas, objetivo) == fuerza_bruta(monedas, objetivo)

1 + 1 + 5 + 10 + 10 + 10 = 37
1 + 1 + 5 + 10 + 20 = 37
2 + 5 + 10 + 10 + 10 = 37
2 + 5 + 10 + 20 = 37


## 14. Punto de encuentro

Dos barcos se mueven en línea recta con velocidad constante: el primero sale de $p_1$ con velocidad $v_1$ y el segundo de $p_2$ con $v_2$. Escribir `encuentro(p1, v1, p2, v2)`, que devuelva el instante $t \ge 0$ y el punto en que coinciden, o `None` si nunca se cruzan. Hay que resolver $(v_1 - v_2)\,t = p_2 - p_1$ en las dos coordenadas a la vez, con `Fraction` para que sea exacto. Comprobarlo con encuentros construidos a propósito.

In [14]:
def encuentro(p1, v1, p2, v2):
    dx, dy = p2[0] - p1[0], p2[1] - p1[1]
    wx, wy = v1[0] - v2[0], v1[1] - v2[1]
    if wx == wy == 0:
        return (Fraction(0), tuple(map(Fraction, p1))) if dx == dy == 0 else None
    t = Fraction(dx) / wx if wx else Fraction(dy) / wy
    if t < 0 or wx * t != dx or wy * t != dy:
        return None
    return t, (p1[0] + v1[0] * t, p1[1] + v1[1] * t)


def mostrar(resultado):
    if resultado is None:
        return "no se cruzan nunca"
    t, (x, y) = resultado
    return f"se encuentran en t = {t} en ({x}, {y})"


casos = {
    "de frente": ((0, 0), (2, 1), (10, 5), (-3, -1.5)),
    "uno alcanza al otro": ((0, 0), (3, 0), (4, 0), (1, 0)),
    "paralelos": ((0, 0), (1, 1), (0, 2), (1, 1)),
    "se cruzan las rutas, no los barcos": ((0, 0), (1, 0), (5, -5), (0, 2)),
    "se alejan": ((0, 0), (-1, 0), (3, 0), (1, 0)),
    "ya juntos y quietos": ((2, 2), (0, 0), (2, 2), (0, 0)),
}
for nombre, datos in casos.items():
    print(f"{nombre:<36} {mostrar(encuentro(*datos))}")

assert encuentro(*casos["de frente"]) == (2, (4, 2))
assert encuentro(*casos["uno alcanza al otro"])[0] == 2
assert all(encuentro(*casos[k]) is None for k in ("paralelos", "se cruzan las rutas, no los barcos", "se alejan"))

random.seed(14)
for _ in range(1000):
    t = Fraction(random.randint(0, 40), random.randint(1, 8))
    punto = (random.randint(-20, 20), random.randint(-20, 20))
    v1, v2 = [(random.randint(-5, 5), random.randint(-5, 5)) for _ in range(2)]
    if v1 == v2:
        continue
    p1 = (punto[0] - v1[0] * t, punto[1] - v1[1] * t)
    p2 = (punto[0] - v2[0] * t, punto[1] - v2[1] * t)
    assert encuentro(p1, v1, p2, v2) == (t, punto)

de frente                            se encuentran en t = 2 en (4, 2)
uno alcanza al otro                  se encuentran en t = 2 en (6, 0)
paralelos                            no se cruzan nunca
se cruzan las rutas, no los barcos   no se cruzan nunca
se alejan                            no se cruzan nunca
ya juntos y quietos                  se encuentran en t = 0 en (2, 2)


## 15. Truco o trato

Un grupo de personas (nombre, edad y altura en cm) pide "truco" o "trato". Con truco reciben un susto por cada 2 letras del nombre, 2 más si su edad es par y 3 por cada 100 cm de altura; con trato, un dulce por cada letra del nombre, uno por cada 3 años hasta los 10 y 2 por cada 50 cm. Escribir `truco_o_trato(opcion, personas)`, que devuelva lo que recibe cada persona elegido al azar de una lista, y comprobar que el número de elementos sigue las reglas.

In [15]:
SUSTOS = ["una araña", "un murciélago", "una calabaza que ríe", "un fantasma", "una lechuza", "un esqueleto", "una bruja"]
DULCES = ["una piruleta", "un bombón", "una gominola", "un caramelo de miel de palma", "un turrón de gofio", "un bienmesabe"]


def letras(nombre):
    return sum(c.isalpha() for c in nombre)


def cuantos(opcion, persona):
    nombre, edad, altura = persona
    if opcion == "truco":
        return letras(nombre) // 2 + (2 if edad % 2 == 0 else 0) + 3 * (altura // 100)
    if opcion == "trato":
        return letras(nombre) + min(edad, 10) // 3 + 2 * (altura // 50)
    raise ValueError(f"opción desconocida: {opcion!r}")


def truco_o_trato(opcion, personas, rng):
    lista = SUSTOS if opcion == "truco" else DULCES
    return {p[0]: rng.choices(lista, k=cuantos(opcion, p)) for p in personas}


grupo = [("Ayoze", 9, 134), ("Chaxiraxi", 12, 151), ("Yeray", 6, 118), ("Guacimara", 4, 99)]
rng = random.Random(15)
for opcion in ("truco", "trato"):
    reparto = truco_o_trato(opcion, grupo, rng)
    print(opcion.upper())
    for nombre, cosas in reparto.items():
        resumen = ", ".join(f"{n} × {x}" for x, n in Counter(cosas).most_common())
        print(f"  {nombre:<10} {len(cosas):>2}: {resumen}")
    assert all(len(reparto[p[0]]) == cuantos(opcion, p) for p in grupo)

assert [cuantos("truco", p) for p in grupo] == [5, 9, 7, 6]
assert [cuantos("trato", p) for p in grupo] == [12, 18, 11, 12]
try:
    truco_o_trato("caramelos", grupo, rng)
except ValueError as e:
    print("Error:", e)

TRUCO
  Ayoze       5: 2 × una bruja, 1 × una araña, 1 × un esqueleto, 1 × un murciélago
  Chaxiraxi   9: 3 × una bruja, 2 × una lechuza, 2 × un murciélago, 1 × una araña, 1 × una calabaza que ríe
  Yeray       7: 3 × un murciélago, 3 × una bruja, 1 × un esqueleto
  Guacimara   6: 3 × un fantasma, 1 × un murciélago, 1 × una calabaza que ríe, 1 × una araña
TRATO
  Ayoze      12: 4 × un bienmesabe, 3 × una gominola, 2 × un caramelo de miel de palma, 2 × una piruleta, 1 × un bombón
  Chaxiraxi  18: 4 × un bienmesabe, 4 × una gominola, 3 × un bombón, 3 × una piruleta, 2 × un turrón de gofio, 2 × un caramelo de miel de palma
  Yeray      11: 3 × una gominola, 2 × un bombón, 2 × un turrón de gofio, 2 × un bienmesabe, 2 × una piruleta
  Guacimara  12: 4 × una gominola, 4 × un bienmesabe, 2 × una piruleta, 1 × un turrón de gofio, 1 × un bombón
Error: opción desconocida: 'caramelos'
